# MathCursor — Benchmark modèles NER (corpus complet (auto-glob v3->v11 + fr/en))

Teste **XLM-RoBERTa-base**, **DistilBERT multilingue** et **MiniLM multilingue**
sur le corpus enrichi (auto-glob) (v2 + fixtures projet + `²` + anti-FP + keywords math
+ quantificateurs ∀/∃ et lettres V/E + vecteurs/coords v6 + conjonctions v7
+ n-aires formes courtes / iint / iiint / mots-clés nus v8).

Critère d'adoption d'un modèle plus léger :
- F1 ≥ 0.98 sur `test.jsonl`
- F1 ≥ 0.99 sur `extension_v3_fixtures.jsonl` (gold)
- Taille `.onnx` int8 minimale

**Setup** :
1. Créer `MyDrive/mathcursor/` dans ton Google Drive
2. Y déposer tous les `.jsonl` du dossier `data/ner-corpus/` du repo
   (la cellule 1 vérifie la liste exacte et échoue si un fichier manque)
3. Runtime → GPU T4 → Tout exécuter

Durée : ~10-15 min (3 modèles × 4 epochs sur ~7300 lignes).

## 1. Drive + vérification fichiers

In [ ]:
from google.colab import drive
import os, glob

DRIVE_FOLDER = 'mathcursor'

drive.mount('/content/drive')
WORKDIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'

# Obligatoires : splits + gold (hold-out d'eval, JAMAIS dans le train).
REQUIRED = ['train.jsonl', 'val.jsonl', 'test.jsonl', 'regression_v1_gold.jsonl']
for name in REQUIRED:
    path = os.path.join(WORKDIR, name)
    if not os.path.isfile(path):
        raise RuntimeError(f'Fichier manquant : {path}')
    print(f'OK {name}: {os.path.getsize(path)/1024:.1f} Ko')

# Extensions = AUTO-DECOUVERTE (tout extension_*.jsonl) -> ajoutees au train.
EXTENSIONS = sorted(
    os.path.basename(p)
    for p in glob.glob(os.path.join(WORKDIR, 'extension_*.jsonl'))
)
for name in EXTENSIONS:
    print(f'OK {name}: {os.path.getsize(os.path.join(WORKDIR, name))/1024:.1f} Ko (extension)')

print()
print(f'WORKDIR = {WORKDIR} | {len(EXTENSIONS)} extensions')

## 2. Dépendances

In [ ]:
!pip uninstall -y diffusers
!pip install -q -U transformers datasets accelerate seqeval \
    "optimum[onnxruntime]" onnx onnxruntime sentencepiece protobuf

print('\n⚠️  Runtime → Restart session → puis Run all à partir d\'ici')

## 3. Paramètres — 3 candidats

In [ ]:
# MiniLM retiré du benchmark : tokenizer cassé sur l'alignement char→token
# (rate systématiquement la 1ère lettre des spans, ex: "f(x)" → "(x)").
# Indépendant du corpus, inadoptable même si F1 monte. Cf. analyse 30-04.
MODELS = [
    {'name': 'xlm-roberta-base',                       'short': 'xlmr',      'lr': 3e-5},
    {'name': 'distilbert-base-multilingual-cased',     'short': 'distilmult','lr': 5e-5},
]

MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_EPOCHS = 4
WEIGHT_DECAY = 0.01

LABELS = ['O', 'B-MATH', 'I-MATH']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

LOG_DIR = os.path.join(WORKDIR, 'logs-benchmark')
RESULTS_DIR = os.path.join(WORKDIR, 'benchmark-results')
os.makedirs(RESULTS_DIR, exist_ok=True)

## 4. Chargement corpus v3 (v2 + extensions)

On concatène v2 (train) + extensions v3 pour entraîner.
Val et test restent les mêmes que v2 pour comparer directement avec la baseline.

In [ ]:
import json, os, glob
from datasets import Dataset, DatasetDict

# Fold accents comme en prod (AutocorrectNormalizer en amont du NER) : meme
# distribution train/runtime. Mapping 1:1 -> offsets des spans inchanges.
FOLD = str.maketrans(
    'àáâãäåèéêëìíîïòóôõöùúûüçñýÿÀÁÂÃÄÅÈÉÊËÌÍÎÏÒÓÔÕÖÙÚÛÜÇÑÝ',
    'aaaaaaeeeeiiiiooooouuuucnyyAAAAAAEEEEIIIIOOOOOUUUUCNY')

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        rows = [json.loads(line) for line in f]
    for r in rows:
        r['text'] = r['text'].translate(FOLD)
    return rows

def to_dict(examples):
    return {
        'text': [e['text'] for e in examples],
        'spans': [e['spans'] for e in examples],
        'lang': [e['lang'] for e in examples],
    }

train_v8 = load_jsonl(os.path.join(WORKDIR, 'train.jsonl'))
val      = load_jsonl(os.path.join(WORKDIR, 'val.jsonl'))
test     = load_jsonl(os.path.join(WORKDIR, 'test.jsonl'))
# Gold = hold-out d'eval, JAMAIS concatene au train.
gold     = load_jsonl(os.path.join(WORKDIR, 'regression_v1_gold.jsonl'))

# Toutes les extensions (auto-decouvertes) ajoutees au train.
n0 = len(train_v8)
for name in EXTENSIONS:
    ext = load_jsonl(os.path.join(WORKDIR, name))
    train_v8 += ext
    print(f'+ {name}: {len(ext)}')

datasets = DatasetDict({
    'train':      Dataset.from_dict(to_dict(train_v8)),
    'validation': Dataset.from_dict(to_dict(val)),
    'test':       Dataset.from_dict(to_dict(test)),
    'gold':       Dataset.from_dict(to_dict(gold)),
})
print()
print(f'Train: {len(train_v8)} (base {n0} + {len(train_v8)-n0} ext) | Val {len(val)} | Test {len(test)} | Gold {len(gold)}')
print(datasets)

## 5. Tokenisation + alignement BIO (fonction réutilisable par modèle)

In [ ]:
from transformers import AutoTokenizer

def spans_to_char_labels(text, spans):
    labels = ['O'] * len(text)
    for span in spans:
        for i in range(span['start'], span['end']):
            if i < len(labels):
                labels[i] = 'MATH'
    return labels

def build_tokenize_fn(tokenizer):
    def tokenize_and_align(examples):
        tokenized = tokenizer(
            examples['text'],
            truncation=True,
            max_length=MAX_LENGTH,
            return_offsets_mapping=True,
            padding=False,
        )

        all_labels = []
        for i, text in enumerate(examples['text']):
            char_labels = spans_to_char_labels(text, examples['spans'][i])
            offsets = tokenized['offset_mapping'][i]

            token_labels = []
            prev = 'O'
            for (start, end) in offsets:
                if start == end:
                    token_labels.append(-100)
                    continue
                cl = char_labels[start] if start < len(char_labels) else 'O'
                if cl == 'MATH':
                    token_labels.append(LABEL2ID['B-MATH'] if prev != 'MATH' else LABEL2ID['I-MATH'])
                    prev = 'MATH'
                else:
                    token_labels.append(LABEL2ID['O'])
                    prev = 'O'
            all_labels.append(token_labels)

        tokenized['labels'] = all_labels
        tokenized.pop('offset_mapping')
        return tokenized
    return tokenize_and_align

## 6. Métriques

In [ ]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_labels = [[ID2LABEL[l] for l in lab if l != -100] for lab in labels]
    true_preds  = [[ID2LABEL[p] for (p, l) in zip(pr, lab) if l != -100]
                   for pr, lab in zip(predictions, labels)]

    return {
        'precision': precision_score(true_labels, true_preds),
        'recall':    recall_score(true_labels, true_preds),
        'f1':        f1_score(true_labels, true_preds),
    }

## 7. Fonction train_and_eval

Pour chaque modèle : fine-tune, éval sur test + gold, export ONNX, quantize int8,
mesure taille et latence CPU.

In [ ]:
import time
import shutil
from transformers import (
    AutoModelForTokenClassification, TrainingArguments, Trainer,
    DataCollatorForTokenClassification, pipeline,
)
from optimum.onnxruntime import ORTModelForTokenClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

def train_and_eval(model_cfg: dict) -> dict:
    name = model_cfg['name']
    short = model_cfg['short']
    lr = model_cfg['lr']

    print(f'\n{"="*70}\n  {short.upper()} — {name}\n{"="*70}')

    out_pt    = os.path.join(WORKDIR, f'model-pt-{short}')
    out_onnx  = os.path.join(WORKDIR, f'model-onnx-{short}')
    out_quant = os.path.join(WORKDIR, f'model-onnx-int8-{short}')

    tokenizer = AutoTokenizer.from_pretrained(name, add_prefix_space=False)
    tokenize_fn = build_tokenize_fn(tokenizer)
    tokenized = datasets.map(tokenize_fn, batched=True,
                             remove_columns=['text', 'spans', 'lang'])

    model = AutoModelForTokenClassification.from_pretrained(
        name, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
    )

    args = TrainingArguments(
        output_dir=out_pt,
        logging_dir=os.path.join(LOG_DIR, short),
        learning_rate=lr,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        logging_steps=100,
        save_total_limit=1,
        report_to='none',
        fp16=True,
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'],
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    # Évaluation sur test et sur gold (fixtures)
    test_metrics = trainer.predict(tokenized['test']).metrics
    gold_metrics = trainer.predict(tokenized['gold']).metrics

    f1_test = test_metrics.get('test_f1', test_metrics.get('f1', 0))
    f1_gold = gold_metrics.get('test_f1', gold_metrics.get('f1', 0))

    # Sauvegarde PT + export ONNX + quantize
    trainer.save_model(out_pt)
    tokenizer.save_pretrained(out_pt)

    ort = ORTModelForTokenClassification.from_pretrained(out_pt, export=True)
    ort.save_pretrained(out_onnx)
    tokenizer.save_pretrained(out_onnx)

    quantizer = ORTQuantizer.from_pretrained(out_onnx)
    qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
    quantizer.quantize(save_dir=out_quant, quantization_config=qconfig)
    tokenizer.save_pretrained(out_quant)

    # Taille du dossier quantizé (onnx + tokenizer)
    total_size_mb = sum(
        os.path.getsize(os.path.join(out_quant, f))
        for f in os.listdir(out_quant)
    ) / (1024 * 1024)

    # Taille du .onnx seul
    onnx_file = next(f for f in os.listdir(out_quant) if f.endswith('.onnx'))
    onnx_size_mb = os.path.getsize(os.path.join(out_quant, onnx_file)) / (1024 * 1024)

    # Latence CPU sur 5 phrases
    ort_q = ORTModelForTokenClassification.from_pretrained(
        out_quant, file_name=onnx_file, provider='CPUExecutionProvider',
    )
    ner_q = pipeline('token-classification', model=ort_q, tokenizer=tokenizer,
                     aggregation_strategy='simple', device=-1)

    samples = [
        'On a f(x) = 2x + 1',
        "Calculer l'aire x² + y² = r²",
        'Mon frère Sinane est content',
        'soit f(x) = 2x+1 et g(x) = 3x-1, alors f(g(x)) = 6x+1',
        'Bonjour tout le monde',
    ]
    # warm-up
    _ = ner_q(samples[0])
    latencies = []
    for s in samples:
        t0 = time.time()
        ner_q(s)
        latencies.append((time.time() - t0) * 1000)

    return {
        'short': short,
        'name': name,
        'f1_test': float(f1_test),
        'f1_gold': float(f1_gold),
        'onnx_mb': round(onnx_size_mb, 1),
        'total_mb': round(total_size_mb, 1),
        'latency_ms_mean': round(sum(latencies)/len(latencies), 1),
        'latency_ms_max': round(max(latencies), 1),
        'train_time_s': round(train_time, 1),
        'quant_dir': out_quant,
    }

## 8. Boucle sur les 3 modèles

In [ ]:
results = []
for cfg in MODELS:
    try:
        r = train_and_eval(cfg)
        results.append(r)
        # Sauvegarde incrémentale au cas où
        with open(os.path.join(RESULTS_DIR, 'benchmark.json'), 'w') as f:
            json.dump(results, f, indent=2)
    except Exception as e:
        print(f'\n❌ Échec {cfg["short"]}: {e}')
        results.append({'short': cfg['short'], 'error': str(e)})

## 9. Tableau final & recommandation

In [ ]:
print()
print('=' * 90)
print('  BENCHMARK FINAL — corpus complet (auto-glob v3->v11 + fr/en)')
print('=' * 90)
print()

hdr = f'{"modèle":<14} {"F1 test":>9} {"F1 gold":>9} {"onnx Mo":>9} {"total Mo":>10} {"latence ms":>12}'
print(hdr)
print('-' * len(hdr))

for r in results:
    if 'error' in r:
        print(f'{r["short"]:<14} ❌ {r["error"][:60]}')
        continue
    print(f'{r["short"]:<14} {r["f1_test"]:>9.4f} {r["f1_gold"]:>9.4f} '
          f'{r["onnx_mb"]:>9.1f} {r["total_mb"]:>10.1f} '
          f'{r["latency_ms_mean"]:>8.1f} / {r["latency_ms_max"]:<3.0f}')

print()
print("Critère d'adoption : F1 test ≥ 0.98 ET F1 gold ≥ 0.99")
candidates = [r for r in results if 'error' not in r
              and r['f1_test'] >= 0.98 and r['f1_gold'] >= 0.99]
if candidates:
    best = min(candidates, key=lambda r: r['onnx_mb'])
    print()
    print(f'✅ Recommandation : {best["short"]} ({best["name"]})')
    print(f'   F1 test = {best["f1_test"]:.4f}, gold = {best["f1_gold"]:.4f}')
    print(f'   Taille .onnx int8 = {best["onnx_mb"]:.1f} Mo '
          f'(vs ~265 Mo baseline XLM-R)')
else:
    print()
    print('⚠️  Aucun candidat ne passe le critère. Inspecter les misclassifications gold ci-dessous.')

with open(os.path.join(RESULTS_DIR, 'benchmark.json'), 'w') as f:
    json.dump(results, f, indent=2)

## 9b. Analyse — fixtures gold mal classées

Pour chaque modèle, on relance l'inférence sur `extension_v3_fixtures.jsonl`
et on dump les fixtures où les spans prédits diffèrent des spans attendus.

Objectif : voir si les gaps gold viennent (a) d'annotations gold obsolètes
(les briques 0.5.x ont changé l'attendu) ou (b) d'annotations v6 incohérentes
(à corriger dans `build_v6_recent_features.py`).

Détail complet écrit dans `benchmark-results/misclass-gold-<short>.json`.

In [ ]:
print()
print('=' * 90)
print('  ANALYSE GOLD MISCLASSIFICATIONS')
print('=' * 90)

gold_examples = load_jsonl(os.path.join(WORKDIR, 'regression_v1_gold.jsonl'))
print(f'\nGold = regression_v1_gold.jsonl : {len(gold_examples)} cas curés\n')


def expected_set(spans):
    return frozenset((s['start'], s['end']) for s in spans)


def predicted_set(ner_results):
    return frozenset((r['start'], r['end']) for r in ner_results)


for r in results:
    if 'error' in r:
        continue
    short = r['short']
    quant_dir = r['quant_dir']
    print('\n' + '-' * 90)
    print(f'  {short.upper()}  (F1 gold = {r["f1_gold"]:.4f})')
    print('-' * 90)

    # Charge le modèle quantizé pour inférence CPU
    onnx_file = next(f for f in os.listdir(quant_dir) if f.endswith('.onnx'))
    ort_q = ORTModelForTokenClassification.from_pretrained(
        quant_dir, file_name=onnx_file, provider='CPUExecutionProvider',
    )
    tok = AutoTokenizer.from_pretrained(quant_dir, add_prefix_space=False)
    ner_q = pipeline(
        'token-classification', model=ort_q, tokenizer=tok,
        aggregation_strategy='simple', device=-1,
    )

    mismatches = []
    for ex in gold_examples:
        text = ex['text']
        expected = expected_set(ex['spans'])
        preds_raw = ner_q(text) if text else []
        predicted = predicted_set(preds_raw)
        if expected != predicted:
            mismatches.append({
                'text': text,
                'lang': ex.get('lang', ''),
                'expected_spans': sorted(expected),
                'expected_frags': [text[s:e] for s, e in sorted(expected)],
                'predicted_spans': [(int(p['start']), int(p['end'])) for p in preds_raw],
                'predicted_frags': [(text[int(p['start']):int(p['end'])],
                                     float(p['score'])) for p in preds_raw],
            })

    print(f'\n  {len(mismatches)} cas mal classés sur {len(gold_examples)} '
          f'({100*len(mismatches)/len(gold_examples):.1f} %)\n')

    # Affiche les 25 premières mismatches
    for i, m in enumerate(mismatches[:25]):
        print(f'  [{i+1}] {m["text"]!r}')
        if m['expected_frags']:
            print(f'      attendu : {m["expected_frags"]}')
        else:
            print(f'      attendu : (rien)')
        if m['predicted_frags']:
            preds_str = ', '.join(f'{frag!r}@{conf:.2f}' for frag, conf in m['predicted_frags'])
            print(f'      prédit  : {preds_str}')
        else:
            print(f'      prédit  : (rien)')
        print()
    if len(mismatches) > 25:
        print(f'  ... et {len(mismatches) - 25} de plus (cf JSON)')

    # Sauvegarde liste complète
    misclass_path = os.path.join(RESULTS_DIR, f'misclass-gold-{short}.json')
    with open(misclass_path, 'w', encoding='utf-8') as f:
        json.dump(mismatches, f, indent=2, ensure_ascii=False)
    print(f'\n  Détail JSON : {misclass_path}')

print()
print('=' * 90)
print('  Fait.')
print('=' * 90)

## 10. Archive du modèle recommandé

Zip le dossier quantizé du modèle qui sort gagnant (ou XLM-R par défaut).

In [ ]:
valid = [r for r in results if 'error' not in r]
candidates = [r for r in valid if r['f1_test'] >= 0.98 and r['f1_gold'] >= 0.99]

if candidates:
    winner = min(candidates, key=lambda r: r['onnx_mb'])
elif valid:
    winner = max(valid, key=lambda r: r['f1_test'])
else:
    winner = None

if winner:
    archive_base = os.path.join(WORKDIR, f'mathcursor-ner-v7-{winner["short"]}')
    shutil.make_archive(archive_base, 'zip', winner['quant_dir'])
    archive = archive_base + '.zip'
    print(f'Archive : {archive} ({os.path.getsize(archive)/(1024*1024):.1f} Mo)')
else:
    print('Pas de gagnant à archiver')